# Adaptive Beta Sampling for Diffusion Models

This notebook runs inference for Adaptive Beta Sampling using:
- Stable Diffusion v1.5
- Beta-DDIM scheduling
- FFT-based probe features
- semantic text embeddings
- a trained MLP loaded from Hugging Face

In [ ]:
!pip -q install diffusers transformers accelerate scipy ftfy safetensors
!pip -q install sentence-transformers huggingface_hub joblib

In [ ]:
!git clone https://github.com/ebad426623/adaptive-beta-sampling.git

In [ ]:
import sys
import matplotlib.pyplot as plt

sys.path.append("/content/adaptive-beta-sampling")

from src.adaptive_sampling import AdaptiveBetaSampling

In [ ]:
adaptive_sampler = AdaptiveBetaSampling(
    hf_repo_id="ebad426623/adaptive-beta-sampling",
    sd_model_id="runwayml/stable-diffusion-v1-5",
)

In [ ]:
prompt = "A futuristic city with flying cars at sunset"
seed = 42

result = adaptive_sampler.compare(prompt_text=prompt, seed=seed)

# Extract values
alpha_adapt = result["adaptive"]["alpha"]
beta_adapt = result["adaptive"]["beta"]

default_alpha = adaptive_sampler.config["default_alpha"]
default_beta = adaptive_sampler.config["default_beta"]

plt.figure(figsize=(15, 4))

# Uniform
plt.subplot(1, 3, 1)
plt.imshow(result["uniform"])
plt.title("Uniform\n(α=1.0, β=1.0)")
plt.axis("off")

# Fixed
plt.subplot(1, 3, 2)
plt.imshow(result["fixed"])
plt.title(f"Fixed Beta\n(α={default_alpha}, β={default_beta})")
plt.axis("off")

# Adaptive
plt.subplot(1, 3, 3)
plt.imshow(result["adaptive"]["image"])
plt.title(f"Adaptive\n(α={alpha_adapt}, β={beta_adapt})")
plt.axis("off")

plt.tight_layout()
plt.show()

# Print info
print("Prompt:", prompt)
print("Predicted label:", result["adaptive"]["label"])
print("Predicted region:", result["adaptive"]["region"])
print("Confidence:", round(result["adaptive"]["confidence"], 4))
print("Selected Beta:", (alpha_adapt, beta_adapt))